# KD-ReID — Colab 執行器

**程式碼以 GitHub 為準，Colab 只負責「拉最新程式 → 訓練」。** 不要在 Colab 裡改 `kdreid/` 的程式碼（runtime 重置就會消失）；要改就在 Claude / 本機改完 push，這裡重新執行第 3 步就會 `git pull`。

每次開新的 runtime 時從上到下執行一次。

**事前準備（只需做一次）**
1. Runtime → Change runtime type → **GPU**（T4 即可，A100/L4 更快）
2. 資料集 zip 放在 Drive `MyDrive/KD-ReID/datasets/`（檔名不限，例如 `archive.zip`）
3. 這是私有 repo：在 Colab 左側 🔑 **Secrets** 新增 `GH_TOKEN`（GitHub fine-grained token，只給 `cts0424/KD-ReID` 的 Contents read 權限），並開啟 Notebook access

## 1. 檢查 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. 掛載 Google Drive（存放資料集壓縮檔與 checkpoints）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/KD-ReID/outputs /content/drive/MyDrive/KD-ReID/datasets

## 3. Clone repo 並安裝

Colab 已內建 CUDA 版的 torch，`pip install -e .` 不會重裝它。

In [ ]:
import os
from google.colab import userdata
token = userdata.get('GH_TOKEN')
REPO = '/content/KD-ReID'
if not os.path.exists(REPO):
    !git clone https://{token}@github.com/cts0424/KD-ReID.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip install -q -e ".[dev,log]"

## 4. 把資料集複製到本機磁碟並解壓

直接從 Drive 讀上萬張小圖會非常慢，所以訓練時一律從 `/content/data` 讀。程式會自動在 `/content/data` 底下找到 `bounding_box_train` 所在的資料夾。

In [ ]:
import glob, os, subprocess
DS_DIR = '/content/drive/MyDrive/KD-ReID/datasets'
zips = sorted(glob.glob(f'{DS_DIR}/*.zip'))
assert zips, f'No .zip found in {DS_DIR}'
os.makedirs('/content/data', exist_ok=True)
if not glob.glob('/content/data/**/bounding_box_train', recursive=True):
    for z in zips:
        print('unzipping', z)
        subprocess.run(['cp', z, '/content/'], check=True)
        subprocess.run(['unzip', '-q', '-o', f'/content/{os.path.basename(z)}', '-d', '/content/data'], check=True)
print(*glob.glob('/content/data/**/bounding_box_train', recursive=True), sep='\n')

## 5. Smoke test（約 10 秒，確認環境正常）

In [ ]:
!python -m pytest -q

## 6. 訓練

依序執行：
1. **Teacher**（ResNet-50）→ `outputs/teacher_r50/best.pth`
2. **Student baseline**（ResNet-18，無蒸餾）→ 比較基準
3. **Student + KD**（ResNet-18 ← ResNet-50）

輸出寫在 Drive 上，斷線後重新執行同一行會從 `last.pth` 自動接續。

In [ ]:
!python tools/train.py --config configs/teacher_r50.yaml

In [ ]:
!python tools/train.py --config configs/student_r18_baseline.yaml

In [ ]:
!python tools/train.py --config configs/student_r18_kd.yaml

## 7. OSNet x0.25 學生模型（`docs/plan_osnet.md`）

**第一次執行**會從 torchreid model zoo 下載 OSNet ImageNet 權重到 Drive `KD-ReID/pretrained/`，之後直接重用。
若出現 gdown 配額錯誤（Google Drive 下載太多次），到 torchreid model zoo 手動下載 `osnet_x0_25_imagenet.pth` 放進該資料夾即可。

### 階段 1：強 baseline（無 KD）—— 1a 先跑，其餘看 1a 結果再決定

In [ ]:
!python tools/train.py --config configs/osnet/1a_baseline.yaml

In [ ]:
# 已完成：1a 64.3、1c 71.4、1e 77.0（= B*）。以下為暫緩的補充實驗：
# !python tools/train.py --config configs/osnet/1d_240ep.yaml
# !python tools/train.py --config configs/osnet/1b_embed128.yaml
# !python tools/train.py --config configs/osnet/1s_scratch.yaml

### 階段 2：直接蒸餾 R50 → OSNet x0.25（以 B* 為底）

In [ ]:
# 都以 B*（1e 配方，240 ep）為底；每個約 3–3.5 小時，依序跑
!python tools/train.py --config configs/osnet/2a_kd_r50.yaml
!python tools/train.py --config configs/osnet/2c_kd_r50_at.yaml
!python tools/train.py --config configs/osnet/2d_kd_r50_logit.yaml
# !python tools/train.py --config configs/osnet/2b_kd_r50_feat.yaml   # 優先度最低

## 8. 部署評估

架構比較（隨機權重即可量大小與速度），以及所有已訓練模型（讀每個 output_dir 的 `best.pth`）。
CPU 延遲以 **ONNX Runtime**（`ort_cpu_*`）為準；PyTorch eager 會低估 OSNet 的速度優勢。

In [ ]:
!pip install -q onnx onnxruntime onnxscript
!python tools/benchmark.py --models resnet50 resnet18 osnet_x1_0:512 osnet_x0_25:512 \
    --onnx-dir /content/onnx --out /content/drive/MyDrive/KD-ReID/benchmark_arch.json

## 9. 查看訓練日誌

每個 output_dir 都有 `log.txt`、`config.yaml`、`last.pth`、`best.pth`。**論文報告用最後一個 epoch 的數字**（log 最後的 `Final (epoch N)` 那行）。

In [ ]:
!tail -n 5 /content/drive/MyDrive/KD-ReID/outputs/osnet/1a_baseline/log.txt